In [4]:
print(1,2,3)

1 2 3


In [5]:
from pathlib import Path
from urllib.request import urlretrieve

PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"

files = {
    "ingest.py": f"{PREFIX}/01-agentic-rag/code/ingest.py",
    "rag_helper.py": f"{PREFIX}/01-agentic-rag/code/rag_helper.py",
    "evaluation_utils.py": f"{PREFIX}/04-evaluation/code/evaluation_utils.py",
}

for filename, url in files.items():
    urlretrieve(url, filename)
    print(f"Heruntergeladen: {filename}")

Heruntergeladen: ingest.py
Heruntergeladen: rag_helper.py
Heruntergeladen: evaluation_utils.py


In [6]:
from ingest import load_faq_data
from evaluation_utils import llm_structured, calc_price

In [7]:
from ingest import load_faq_data
documents = load_faq_data()

In [8]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

118

In [9]:
documents = documents_llm

In [10]:
doc = documents[0]
doc["doc_id"] = doc.pop("id")
print(doc["question"])
print(doc["answer"])

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [11]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [12]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [13]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [14]:
import json

user_prompt = json.dumps(doc)

In [15]:
user_prompt

'{"course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions.", "doc_id": "74eb249bbf"}'

In [16]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [17]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [18]:
response.output_parsed.questions


['I just found this course — can I still join, or is it too late?',
 'If I start now, is it still possible to get a certificate?',
 'Do I need to finish and submit the project before submissions close to get certified?',
 'Is late enrollment allowed in this course, or only if I don’t care about the certificate?',
 'What’s the deadline for project submission if I want the course certificate?']

In [19]:
doc

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [20]:
len(documents)

118

In [21]:
from evaluation_utils import llm_structured

In [22]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course—am I still able to join now, or is it too late?', 'If I start the course late, can I still get the certificate somehow?', 'Do I need to enroll at a certain time, or can I join after the course has already started?', 'I missed the beginning of the course; can I still take part and complete it?', 'Is it okay to join the course now, and what do I need to do if I want the certificate?']


In [23]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=111, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=318)

In [24]:
from evaluation_utils import calc_price

In [25]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525,
 'output_cost': 0.0004995,
 'total_cost': 0.0006547500000000001}

In [26]:
records = []

for q in result.questions:
    records = []

    for q in result.questions:
        records.append({
            "question": q,
            "document": doc.get("doc_id", doc.get("id"))
        })

    records

records

[{'question': 'I just found this course—am I still able to join now, or is it too late?',
  'document': '74eb249bbf'},
 {'question': 'If I start the course late, can I still get the certificate somehow?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to enroll at a certain time, or can I join after the course has already started?',
  'document': '74eb249bbf'},
 {'question': 'I missed the beginning of the course; can I still take part and complete it?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to join the course now, and what do I need to do if I want the certificate?',
  'document': '74eb249bbf'}]

In [27]:
import pandas as pd

In [28]:
pd.DataFrame(records)

,question,document
0,I just found this course—am I still able to jo...,74eb249bbf
1,"If I start the course late, can I still get th...",74eb249bbf
2,"Do I need to enroll at a certain time, or can ...",74eb249bbf
3,I missed the beginning of the course; can I st...,74eb249bbf
4,"Is it okay to join the course now, and what do...",74eb249bbf


### Generating Ground Truth for All Documents

In [29]:
from evaluation_utils import llm_structured_retry

In [30]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [31]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []
    doc_id = doc.get("doc_id", doc.get("id"))

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc_id
        })

    return results, usage

In [32]:
generate_ground_truth(doc)

([{'question': 'I just found this course—can I still sign up and start now, or is it too late?',
   'document': '74eb249bbf'},
  {'question': 'If I join the course late, do I still have a chance to get a certificate?',
   'document': '74eb249bbf'},
  {'question': 'What’s the deadline for submitting the project if I want the certificate?',
   'document': '74eb249bbf'},
  {'question': 'Can new students still participate, even though the course has already started?',
   'document': '74eb249bbf'},
  {'question': 'If I’m late to the course, what do I need to do to be eligible for the certificate?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=104, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=311))

In [33]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    def generate_ground_truth(doc):
        user_prompt = json.dumps(doc)

        out, usage = llm_structured_retry(
            openai_client,
            data_gen_instructions,
            user_prompt,
            Questions
        )

        results = []
        doc_id = doc.get("doc_id", doc.get("id"))

        for q in out.questions:
            results.append({
                "question": q,
                "document": doc_id
            })

        return results, usage
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

Parallel processing

Running the calls one after another wastes most of the time waiting on the network. Each request just sits there until OpenAI responds, so we can fire several at once and wait on them together. We process the documents in parallel and track progress while the requests run.

One caution: don't open too many connections at once, or you'll hit the provider's rate limits. Five or six workers is a safe default here.

Import ThreadPoolExecutor:

In [34]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [35]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/118 [00:00<?, ?it/s]

In [36]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

590

In [37]:
ground_truth[10]

{'question': 'Where can I watch the Office Hours or live workshop stream if I’m a student?',
 'document': '489dd1c9d9'}

Calculate the total cost:



In [38]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.09209099999999999

In [39]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.09209099999999999

In [40]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [41]:
#df_ground_truth.to_csv("/Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh-1/04_Evaluation/01_data_generate.ipynb/data/ground_truth.csv", index=False)

In [42]:
len(df_ground_truth)

590

In [43]:
from pathlib import Path

Path("data").mkdir(exist_ok=True)
df_ground_truth.to_csv("data/ground_truth.csv", index=False)